In [54]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# pd.set_option('display.max_colwidth', None)  # Display full content of each column
# pd.set_option('display.max_columns', None)   # Display all columns
# pd.set_option('display.width', 5000)         # Set display width

In [55]:

df_sms = pd.read_excel(r'C:\Users\yashs\Desktop\Journey\SpendWise\ml_preprocessing\CSVS\captured_sms_cleaned.xlsx')
df_bank = pd.read_excel('2026_Clean.xlsx')


In [56]:
df_sms.shape
df_bank.shape

(500, 12)

In [85]:
df_sms.columns

Index(['id', 'sender', 'body', 'timestamp_ms', 'timestamp_human'], dtype='object')

In [58]:
# # Keep only distinct rows based on the 'body' column
# df = df.drop_duplicates(subset=['body']).reset_index(drop=True)
# print(f"After removing duplicate bodies: {len(df)} rows remaining")

# df_sms = df_sms.drop(columns=['is_financial', 'amount', 'direction', 'device_id', 'bank','recipient','upi_id'])
# df_sms.to_excel("CSVS\\captured_sms_cleaned.xlsx", index=False)

In [59]:
print(df_sms.head())

                                     id       sender  \
0  da45603f-fcaa-4d2e-a4ec-e228abc7d2ff  JD-SBIUPI-S   
1  aef8fa49-3cc8-4f56-a222-8cb6f801c828  AD-SBIUPI-S   
2  8a3353c3-35e6-4004-aa16-ae7b054748c4  JD-SBIUPI-S   
3  b047d419-0f35-45fd-896e-251e0bd524af    AZ-AIRINF   
4  de10db4b-1ab2-4c0b-80dc-c40a331e9404  JK-SBIUPI-S   

                                                body   timestamp_ms  \
0  Dear UPI user A/C X7686 debited by 120 on date...  1767283633363   
1  Dear UPI user A/C X7686 debited by 128 on date...  1767794208425   
2  Dear UPI user A/C X7686 debited by 138 on date...  1767958717651   
3  Hi, Payment of Rs. 33.0 has failed for your Ai...  1768213448975   
4  Dear UPI user A/C X7686 debited by 15.00 on da...  1768879517519   

       timestamp_human  
0  2026-01-01 21:37:13  
1  2026-01-07 19:26:48  
2  2026-01-09 17:08:37  
3  2026-01-12 15:54:08  
4  2026-01-20 08:55:17  


In [60]:
print(df_sms.dtypes)

print(df_sms['timestamp_ms'].head())

print(df_sms['timestamp_human'].head())

id                 object
sender             object
body               object
timestamp_ms       object
timestamp_human    object
dtype: object
0    1767283633363
1    1767794208425
2    1767958717651
3    1768213448975
4    1768879517519
Name: timestamp_ms, dtype: object
0    2026-01-01 21:37:13
1    2026-01-07 19:26:48
2    2026-01-09 17:08:37
3    2026-01-12 15:54:08
4    2026-01-20 08:55:17
Name: timestamp_human, dtype: object



# PHASE 1: SMS PROFILING & NORMALIZATION
**Objective:** Normalize raw SMS, remove duplicates, generate profile statistics

**Output:** `sms_profile.csv` with deduplicated, normalized SMS


In [61]:
import json
import re
from datetime import datetime, timedelta
from collections import Counter
import hashlib

# PHASE 1: SMS PROFILING FUNCTIONS
def normalize_sms(df_sms_raw):
    """
    Normalize SMS dataset
    """
    df = df_sms_raw.copy()

    # --------------------------------------------------
    # Step 1: Create datetime column
    # --------------------------------------------------
    if 'timestamp_ms' in df.columns:
        df['timestamp_ms'] = pd.to_numeric(
            df['timestamp_ms'],
            errors='coerce'
        )

        df['datetime'] = pd.to_datetime(
            df['timestamp_ms'],
            unit='ms',
            errors='coerce',
            utc=True
        ).dt.tz_convert('Asia/Kolkata')

    else:
        df['datetime'] = pd.NaT

    # --------------------------------------------------
    # Step 2: Fallback to timestamp_human
    # --------------------------------------------------
    if 'timestamp_human' in df.columns:

        mask = df['datetime'].isna()

        if mask.any():
            fallback_dates = pd.to_datetime(
                df.loc[mask, 'timestamp_human'],
                errors='coerce'
            )

            fallback_dates = fallback_dates.dt.tz_localize(
                'Asia/Kolkata',
                nonexistent='NaT',
                ambiguous='NaT'
            )

            df.loc[mask, 'datetime'] = fallback_dates

    # --------------------------------------------------
    # Step 3: Force datetime dtype
    # --------------------------------------------------
    df['datetime'] = pd.to_datetime(
        df['datetime'],
        errors='coerce'
    )

    # --------------------------------------------------
    # Step 4: Extract date & time
    # --------------------------------------------------
    df['date'] = df['datetime'].dt.normalize()

    df['time'] = df['datetime'].dt.strftime('%H:%M:%S')

    # --------------------------------------------------
    # Step 5: Normalize sender
    # --------------------------------------------------
    if 'sender' in df.columns:
        df['sender'] = (
            df['sender']
            .astype(str)
            .str.upper()
            .str.strip()
        )

    # --------------------------------------------------
    # Step 6: Clean body
    # --------------------------------------------------
    if 'body' in df.columns:
        df['body'] = (
            df['body']
            .astype(str)
            .str.replace(r'\s+', ' ', regex=True)
            .str.strip()
        )

    print(
        f"✓ Date range: "
        f"{df['date'].min().date()} to "
        f"{df['date'].max().date()}"
    )

    print(
        df[['sender', 'date', 'time']]
        .head(3)
        .to_string(index=False)
    )

    return df
def deduplicate_sms(df_normalized):
    """
    Step 2: Deduplicate by (date, sender, body_hash)
    Keep first occurrence
    """
    initial_count = len(df_normalized)
    
    # Create hash of body for comparison
    df_normalized['body_hash'] = df_normalized['body'].apply(
        lambda x: hashlib.md5(str(x).encode()).hexdigest()
    )
    
    # Deduplicate
    df_dedup = df_normalized.drop_duplicates(
        subset=['date', 'sender', 'body_hash'],
        keep='first'
    ).reset_index(drop=True)
    
    removed = initial_count - len(df_dedup)
    print(f"✓ Removed {removed} duplicates ({removed/initial_count*100:.1f}%)")
    print(f"✓ {len(df_dedup)} unique SMS remaining")
    
    return df_dedup

def profile_sms(df_normalized):
    """
    Step 3: Generate SMS profile statistics
    """
    profile = {
        'total_sms': len(df_normalized),
        'date_range': {
            'start': df_normalized['date'].min().strftime('%Y-%m-%d') if pd.notna(df_normalized['date'].min()) else 'N/A',
            'end': df_normalized['date'].max().strftime('%Y-%m-%d') if pd.notna(df_normalized['date'].max()) else 'N/A'
        },
        'senders': {
            'unique_count': df_normalized['sender'].nunique(),
            'top_10': df_normalized['sender'].value_counts().head(10).to_dict()
        },
        'message_lengths': {
            'min': int(df_normalized['body'].str.len().min()),
            'max': int(df_normalized['body'].str.len().max()),
            'mean': float(df_normalized['body'].str.len().mean())
        },
        'data_quality': {
            'null_dates': int(df_normalized['date'].isna().sum()),
            'null_senders': int(df_normalized['sender'].isna().sum()),
            'null_bodies': int(df_normalized['body'].isna().sum())
        }
    }
    
    return profile

# Execute Phase 1
print("="*60)
print("PHASE 1: SMS PROFILING & NORMALIZATION")
print("="*60)

df_sms_normalized = normalize_sms(df_sms)
df_sms_profile = deduplicate_sms(df_sms_normalized)
sms_stats = profile_sms(df_sms_profile)

print(f"\n📊 SMS Profile Statistics:")
print(json.dumps(sms_stats, indent=2, default=str))


PHASE 1: SMS PROFILING & NORMALIZATION
✓ Date range: 2026-01-01 to 2026-05-10
     sender                      date     time
JD-SBIUPI-S 2026-01-01 00:00:00+05:30 21:37:13
AD-SBIUPI-S 2026-01-07 00:00:00+05:30 19:26:48
JD-SBIUPI-S 2026-01-09 00:00:00+05:30 17:08:37
✓ Removed 0 duplicates (0.0%)
✓ 1145 unique SMS remaining

📊 SMS Profile Statistics:
{
  "total_sms": 1145,
  "date_range": {
    "start": "2026-01-01",
    "end": "2026-05-10"
  },
  "senders": {
    "unique_count": 285,
    "top_10": {
      "NAN": 666,
      "JD-SBIUPI-S": 14,
      "\u3084\u304f\u3056": 13,
      "LINKEDIN": 11,
      "DIP JAIN": 9,
      "X VAY Z": 9,
      "AX-SBIUPI-S": 8,
      "AZ-AIRTEL-P": 7,
      "AD-SBIUPI-S": 7,
      "UPDATING OFFLINE MAPS": 6
    }
  },
  "message_lengths": {
    "min": 1,
    "max": 1024,
    "mean": 119.22794759825328
  },
  "data_quality": {
    "null_dates": 3,
    "null_senders": 0,
    "null_bodies": 0
  }
}


In [62]:

# VERIFY: Show extracted dates
print("\n📅 DATE EXTRACTION VERIFICATION")
print("="*60)
print("\nSample SMS with extracted dates:")
display(df_sms_profile[['sender', 'body', 'date', 'time', 'datetime']].head(10))

print(f"\nDate Statistics:")
print(f"  Earliest: {df_sms_profile['date'].min()}")
print(f"  Latest: {df_sms_profile['date'].max()}")
print(f"  Date Range: {(df_sms_profile['date'].max() - df_sms_profile['date'].min()).days} days")
print(f"  Null dates: {df_sms_profile['date'].isna().sum()}")

# Check time distribution
print(f"\nTime Distribution (samples):")
print(df_sms_profile['time'].head(20))



📅 DATE EXTRACTION VERIFICATION

Sample SMS with extracted dates:


,sender,body,date,time,datetime
0,JD-SBIUPI-S,Dear UPI user A/C X7686 debited by 120 on date...,2026-01-01 00:00:00+05:30,21:37:13,2026-01-01 21:37:13.363000+05:30
1,AD-SBIUPI-S,Dear UPI user A/C X7686 debited by 128 on date...,2026-01-07 00:00:00+05:30,19:26:48,2026-01-07 19:26:48.425000+05:30
2,JD-SBIUPI-S,Dear UPI user A/C X7686 debited by 138 on date...,2026-01-09 00:00:00+05:30,17:08:37,2026-01-09 17:08:37.651000+05:30
3,AZ-AIRINF,"Hi, Payment of Rs. 33.0 has failed for your Ai...",2026-01-12 00:00:00+05:30,15:54:08,2026-01-12 15:54:08.975000+05:30
4,JK-SBIUPI-S,Dear UPI user A/C X7686 debited by 15.00 on da...,2026-01-20 00:00:00+05:30,08:55:17,2026-01-20 08:55:17.519000+05:30
5,VM-SBIINB-S,"Dear Customer, Your a/c no. XXXXXXXX7686 is cr...",2026-01-20 00:00:00+05:30,21:05:36,2026-01-20 21:05:36.196000+05:30
6,JK-SBIUPI-S,Dear UPI user A/C X7686 debited by 29.00 on da...,2026-01-23 00:00:00+05:30,08:22:32,2026-01-23 08:22:32.585000+05:30
7,JD-SBIUPI-S,Dear UPI user A/C X7686 debited by 12.00 on da...,2026-01-23 00:00:00+05:30,08:28:47,2026-01-23 08:28:47.351000+05:30
8,JK-SBIUPI-S,Dear UPI user A/C X7686 debited by 68.00 on da...,2026-01-29 00:00:00+05:30,17:53:23,2026-01-29 17:53:23.592000+05:30
9,AD-SBIPSG-S,"Dear Customer, INR 75,000.00 credited to your ...",2026-02-25 00:00:00+05:30,14:09:13,2026-02-25 14:09:13.617000+05:30



Date Statistics:
  Earliest: 2026-01-01 00:00:00+05:30
  Latest: 2026-05-10 00:00:00+05:30
  Date Range: 129 days
  Null dates: 3

Time Distribution (samples):
0     21:37:13
1     19:26:48
2     17:08:37
3     15:54:08
4     08:55:17
5     21:05:36
6     08:22:32
7     08:28:47
8     17:53:23
9     14:09:13
10    19:49:24
11    09:21:57
12    09:22:00
13    09:24:01
14    09:38:41
15    10:09:01
16    06:58:17
17    14:33:22
18    16:22:08
19    16:22:26
Name: time, dtype: object



# PHASE 2: REGEX LIBRARY & ENTITY EXTRACTION
**Objective:** Extract financial entities using regex patterns

**Entities:** Amount, Account, UPI, Transaction ID, Merchant, Direction, Balance


In [75]:
import re
import pandas as pd

# =====================================================
# REGEX LIBRARY V2
# =====================================================

regex_library = {

    'AMOUNTS': {

        # Rs.1000.00
        'rs_prefix': re.compile(
            r'(?:Rs\.?|INR|₹)\s*([\d,]+(?:\.\d{1,2})?)',
            re.I
        ),

        # debited by 120
        'debited_by': re.compile(
            r'debited\s+by\s+([\d,]+(?:\.\d{1,2})?)',
            re.I
        ),

        # credited by 540
        'credited_by': re.compile(
            r'credited\s+by\s+(?:Rs\.?|INR|₹)?\s*([\d,]+(?:\.\d{1,2})?)',
            re.I
        ),

        # withdrawn Rs.1000
        'withdrawn': re.compile(
            r'withdrawn.*?(?:Rs\.?|INR|₹)\s*([\d,]+(?:\.\d{1,2})?)',
            re.I
        )
    },

    'ACCOUNTS': {

        # A/C X7686
        'sbi_account': re.compile(
            r'A[/]?C\s+X+(\d{4})',
            re.I
        ),

        # A/c XXXXX7686
        'masked_account': re.compile(
            r'A[/]?c\s+X+(\d{4})',
            re.I
        ),

        # x7686
        'short_account': re.compile(
            r'\bx(\d{4})\b',
            re.I
        ),

        # ending 7686
        'ending': re.compile(
            r'ending\s+(\d{4})',
            re.I
        )
    },

    'TRANSACTION_ID': {

        'refno': re.compile(
            r'Ref(?:erence)?\s*No\.?\s*(\d+)',
            re.I
        ),

        'refno2': re.compile(
            r'Refno\s*(\d+)',
            re.I
        ),

        'refhash': re.compile(
            r'Ref#\s*(\d+)',
            re.I
        ),

        'transaction_number': re.compile(
            r'Transaction\s+Number\s+(\d+)',
            re.I
        ),

        'utr': re.compile(
            r'UTR\s+([A-Z0-9]+)',
            re.I
        )
    },

    'DIRECTION': {

        'debit': re.compile(
            r'\b(debited|withdrawn|paid|sent|charged)\b',
            re.I
        ),

        'credit': re.compile(
            r'\b(credited|received|refund)\b',
            re.I
        )
    }
}

In [76]:
import re
import pandas as pd

# =====================================================
# REGEX LIBRARY V2
# =====================================================

regex_library = {

    'AMOUNTS': {

        # Rs.1000.00
        'rs_prefix': re.compile(
            r'(?:Rs\.?|INR|₹)\s*([\d,]+(?:\.\d{1,2})?)',
            re.I
        ),

        # debited by 120
        'debited_by': re.compile(
            r'debited\s+by\s+([\d,]+(?:\.\d{1,2})?)',
            re.I
        ),

        # credited by 540
        'credited_by': re.compile(
            r'credited\s+by\s+(?:Rs\.?|INR|₹)?\s*([\d,]+(?:\.\d{1,2})?)',
            re.I
        ),

        # withdrawn Rs.1000
        'withdrawn': re.compile(
            r'withdrawn.*?(?:Rs\.?|INR|₹)\s*([\d,]+(?:\.\d{1,2})?)',
            re.I
        )
    },

    'ACCOUNTS': {

        # A/C X7686
        'sbi_account': re.compile(
            r'A[/]?C\s+X+(\d{4})',
            re.I
        ),

        # A/c XXXXX7686
        'masked_account': re.compile(
            r'A[/]?c\s+X+(\d{4})',
            re.I
        ),

        # x7686
        'short_account': re.compile(
            r'\bx(\d{4})\b',
            re.I
        ),

        # ending 7686
        'ending': re.compile(
            r'ending\s+(\d{4})',
            re.I
        )
    },

    'TRANSACTION_ID': {

        'refno': re.compile(
            r'Ref(?:erence)?\s*No\.?\s*(\d+)',
            re.I
        ),

        'refno2': re.compile(
            r'Refno\s*(\d+)',
            re.I
        ),

        'refhash': re.compile(
            r'Ref#\s*(\d+)',
            re.I
        ),

        'transaction_number': re.compile(
            r'Transaction\s+Number\s+(\d+)',
            re.I
        ),

        'utr': re.compile(
            r'UTR\s+([A-Z0-9]+)',
            re.I
        )
    },

    'DIRECTION': {

        'debit': re.compile(
            r'\b(debited|withdrawn|paid|sent|charged)\b',
            re.I
        ),

        'credit': re.compile(
            r'\b(credited|received|refund)\b',
            re.I
        )
    }
}

In [77]:
def extract_merchant(text):

    if not isinstance(text, str):
        return None

    patterns = [

        # SBI UPI debit
        r'trf\s+to\s+(.+?)\s+Refno',

        # SBI UPI credit
        r'transfer\s+from\s+(.+?)\s+Ref\s+No',

        # YONO transfer
        r'transfer\s+to\s+(.+?)\s+Ac\s+x',

        # CBS transfer
        r'Transferred\s+to\s+(.+?)\.\s+Avl',

        # Slice
        r'to\s+(.+?)\s+\(UPI\s+Ref'
    ]

    for pattern in patterns:

        match = re.search(pattern, text, re.I)

        if match:

            merchant = match.group(1)

            merchant = re.sub(r'\s+', ' ', merchant)

            merchant = merchant.strip()

            merchant = merchant[:80]

            return merchant

    return None

In [78]:
def extract_account(text):

    if not isinstance(text, str):
        return None

    for pattern in regex_library['ACCOUNTS'].values():

        match = pattern.search(text)

        if match:
            return match.group(1)

    return None

In [79]:
def extract_upi(text):

    if not isinstance(text, str):
        return None

    match = re.search(
        r'([a-zA-Z0-9._-]+@[a-zA-Z]+)',
        text
    )

    if match:
        return match.group(1)

    return None

In [80]:
def extract_transaction_type(text):

    if not isinstance(text, str):
        return None

    text = text.lower()

    if "upi user" in text and "debited" in text:
        return "UPI_DEBIT"

    if "upi user" in text and "credited" in text:
        return "UPI_CREDIT"

    if "imps" in text:
        return "IMPS"

    if "neft" in text:
        return "NEFT"

    if "withdrawn at" in text:
        return "ATM"

    if "yono" in text:
        return "YONO_TRANSFER"

    return "BANK_TRANSACTION"

In [81]:
def extract_entities(df):

    df_entities = pd.DataFrame()

    df_entities['sms_id'] = df.index

    df_entities['amount'] = df['body'].apply(extract_amount)

    df_entities['account_last4'] = df['body'].apply(extract_account)

    df_entities['merchant'] = df['body'].apply(extract_merchant)

    df_entities['direction'] = df['body'].apply(extract_direction)

    df_entities['transaction_id'] = df['body'].apply(extract_transaction_id)

    df_entities['transaction_type'] = df['body'].apply(
        extract_transaction_type
    )

    return df_entities

In [ ]:
df_financial=extract_entities(df_sms)


,sms_id,amount,account_last4,merchant,direction,transaction_id,transaction_type
0,0,120.0,7686,PRADHAN MANTRI B,DR,600132505284,UPI_DEBIT
1,1,128.0,7686,JULFIKAR KHAN,DR,600728010190,UPI_DEBIT
2,2,138.0,7686,SNOW_CREAM_ICE_C,DR,600990095903,UPI_DEBIT
3,3,33.0,None,None,DR,None,BANK_TRANSACTION
4,4,15.0,7686,Compass India Fo,DR,602038413734,UPI_DEBIT
...,...,...,...,...,...,...,...
1140,1140,NaN,None,None,None,None,BANK_TRANSACTION
1141,1141,200.0,None,None,None,None,BANK_TRANSACTION
1142,1142,NaN,None,None,None,None,BANK_TRANSACTION
1143,1143,NaN,None,None,None,None,BANK_TRANSACTION


In [84]:
df_financial.shape

(1145, 7)


# PHASE 3: MERCHANT KNOWLEDGE BASE & SMS CLASSIFICATION
**Objective:** Normalize merchants and classify SMS into 7 categories

**Categories:** TRANSACTION, OTP, PROMOTIONAL, SERVICE_NOTIFICATION, SPAM, FAILED_TRANSACTION, UNKNOWN


In [71]:
# PHASE 3A: MERCHANT KNOWLEDGE BASE

knowledge_base = {
    'merchants': {
        'AMAZON': ['AMZN', 'AMAZON PAY', 'AMAZON.COM', 'AMAZON INDIA'],
        'FLIPKART': ['FK', 'FLIPKART INDIA'],
        'SWIGGY': ['SWIGGY INSTAMART', 'SWIGGY FOOD', 'SWGY'],
        'UBER': ['UBER TECHNOLOGIES', 'UBER EATS'],
        'OLA': ['OLA CABS', 'OLA'],
        'GOOGLEPAY': ['GOOGLE PAY'],
        'PHONEPE': ['PHONEPE'],
        'PAYTM': ['PAYTM'],
        'HDFC': ['HDFC BANK', 'HDFC.COM'],
        'ICICI': ['ICICI BANK'],
        'AXIS': ['AXIS BANK'],
        'SBI': ['STATE BANK'],
    },
    'otp_keywords': ['otp', 'verification', 'password', 'pin', 'code'],
    'promotional_keywords': ['offer', 'discount', 'cashback', 'limited', 'apply', 'approve', 'loan', 'credit card'],
    'service_keywords': ['balance', 'statement', 'maintenance', 'alert', 'update'],
    'failed_keywords': ['failed', 'declined', 'unsuccessful', 'reversed', 'error'],
    'spam_keywords': ['click', 'verify account', 'update details', 'confirm', 'win', 'claim', 'free']
}


def normalize_merchant(merchant_name):
    """Normalize merchant name against knowledge base"""
    if merchant_name is None or pd.isna(merchant_name):
        return None

    merchant_upper = str(merchant_name).upper().strip()
    if merchant_upper == '':
        return None

    for canonical, aliases in knowledge_base['merchants'].items():
        alias_set = {canonical} | {a.upper() for a in aliases}
        if merchant_upper in alias_set:
            return canonical

    for canonical, aliases in knowledge_base['merchants'].items():
        for alias in aliases:
            alias_upper = alias.upper()
            if alias_upper in merchant_upper or merchant_upper in alias_upper:
                return canonical

    return merchant_upper


def _has_keyword(text, keywords):
    if not isinstance(text, str):
        return False
    return any(keyword in text for keyword in keywords)


def classify_sms(text, sender=''):
    """Classify SMS into categories and return a confidence score."""
    if pd.isna(text):
        return 'UNKNOWN', 0.50

    text = str(text)
    text_lower = text.lower()

    if _has_keyword(text_lower, knowledge_base['otp_keywords']):
        if re.search(r'\b\d{4,8}\b', text):
            return 'OTP', 0.95

    if _has_keyword(text_lower, knowledge_base['spam_keywords']):
        return 'SPAM', 0.85

    promo_score = sum(keyword in text_lower for keyword in knowledge_base['promotional_keywords'])
    if promo_score >= 2:
        return 'PROMOTIONAL', 0.85

    amount = extract_amount(text)
    direction = extract_direction(text)
    merchant = extract_merchant(text)

    has_amount = amount is not None
    has_direction = direction is not None
    has_merchant = merchant is not None
    has_failed = _has_keyword(text_lower, knowledge_base['failed_keywords'])
    has_service = _has_keyword(text_lower, knowledge_base['service_keywords'])

    if has_failed and has_amount:
        return 'FAILED_TRANSACTION', 0.90

    if not has_amount and not has_direction and has_service:
        return 'SERVICE_NOTIFICATION', 0.80

    if has_amount and has_direction:
        return 'TRANSACTION', 0.95

    if has_amount and has_merchant:
        return 'TRANSACTION', 0.90

    if has_amount and not any(
        _has_keyword(text_lower, group)
        for group in [knowledge_base['otp_keywords'], knowledge_base['spam_keywords'], knowledge_base['promotional_keywords'], knowledge_base['service_keywords']]
    ):
        return 'TRANSACTION', 0.80

    return 'UNKNOWN', 0.50


def classify_sms_batch(df, text_column='body'):
    """Classify a batch of SMS rows and return classification plus confidence."""
    results = df[text_column].fillna('').astype(str).map(classify_sms)
    df_out = pd.DataFrame(results.tolist(), index=df.index, columns=['classification', 'confidence'])
    return df_out


# Execute Phase 3
print("="*60)
print("PHASE 3: MERCHANT NORMALIZATION & SMS CLASSIFICATION")
print("="*60)

# Normalize merchants
df_entities['merchant_normalized'] = df_entities['merchant'].apply(normalize_merchant)

# Classify SMS
if 'body' not in df_sms_profile.columns:
    raise KeyError("Expected a 'body' column in df_sms_profile for SMS classification.")

df_classification = classify_sms_batch(df_sms_profile, text_column='body')

df_classification['is_financial'] = df_classification['classification'].isin(['TRANSACTION', 'FAILED_TRANSACTION'])

df_entities = pd.concat([df_entities, df_classification], axis=1)

print(f"\n✓ Classified {len(df_entities)} SMS")
print("\nClassification Distribution:")
print(df_entities['classification'].value_counts())

print("\nConfidence Distribution:")
print(df_entities['confidence'].describe())

print("\nFinancial SMS Count:")
print(df_entities['is_financial'].sum())

for category in ['TRANSACTION', 'FAILED_TRANSACTION']:
    sample = df_entities[df_entities['classification'] == category].head(1)
    if len(sample) > 0:
        print(f"\n{category} Sample: {sample.iloc[0]['confidence']:.2f}")


PHASE 3: MERCHANT NORMALIZATION & SMS CLASSIFICATION

✓ Classified 1145 SMS

Classification Distribution:


PHASE 3: MERCHANT NORMALIZATION & SMS CLASSIFICATION

✓ Classified 1145 SMS

Classification Distribution:


ValueError: Grouper for 'classification' not 1-dimensional


# PHASE 4: TRANSACTION MATCHING ENGINE
**Objective:** Match SMS transactions to bank statement entries

**Strategy:** Multi-dimensional matching (amount, date, merchant, direction)


In [65]:

# PHASE 4: TRANSACTION MATCHING

def score_amount_match(sms_amount, bank_amount, tolerance=0.01):
    """Score amount match: 0-1"""
    if pd.isna(sms_amount) or pd.isna(bank_amount):
        return 0.0
    
    if bank_amount == 0:
        return 0.0
    
    diff = abs(sms_amount - bank_amount) / bank_amount
    
    if diff < tolerance:  # Within 1%
        return 0.95
    elif diff < 0.05:  # Within 5%
        return 0.80
    else:
        return 0.0

def score_date_match(sms_date, bank_date, window_days=2):
    """Score date match: 0-1"""
    if pd.isna(sms_date) or pd.isna(bank_date):
        return 0.0
    
    date_diff = abs((sms_date - bank_date).days)
    
    if date_diff == 0:
        return 0.95
    elif date_diff == 1:
        return 0.85
    elif date_diff <= window_days:
        return 0.70
    else:
        return 0.0

def score_merchant_match(sms_merchant, bank_merchant):
    """Score merchant match: 0-1"""
    if pd.isna(sms_merchant) or pd.isna(bank_merchant):
        return 0.5  # Neutral if either missing
    
    sms_m = str(sms_merchant).upper()
    bank_m = str(bank_merchant).upper()
    
    # Exact match
    if sms_m == bank_m:
        return 0.95
    
    # Substring match
    if sms_m in bank_m or bank_m in sms_m:
        return 0.80
    
    return 0.0

def score_direction_match(sms_direction, bank_direction):
    """Score direction match: 0-1"""
    if pd.isna(sms_direction) or pd.isna(bank_direction):
        return 0.5
    
    if sms_direction == bank_direction:
        return 0.95
    else:
        return 0.0

def calculate_match_confidence(sms_row, bank_row):
    """
    Calculate overall match confidence
    Weighted scoring: amount(0.30) + date(0.25) + merchant(0.20) + direction(0.15) + buffer(0.10)
    """
    scores = {
        'amount': score_amount_match(sms_row['amount'], bank_row['Amount']),
        'date': score_date_match(
            pd.to_datetime(sms_row['date']) if 'date' in sms_row.index else None,
            bank_row['Transaction_Date']
        ),
        'merchant': score_merchant_match(sms_row['merchant_normalized'], bank_row.get('Recipient_Name', None)),
        'direction': score_direction_match(sms_row['direction'], bank_row.get('DR/CR_Indicator', None)),
    }
    
    confidence = (
        scores['amount'] * 0.30 +
        scores['date'] * 0.25 +
        scores['merchant'] * 0.20 +
        scores['direction'] * 0.15 +
        0.10  # Base confidence
    )
    
    return min(1.0, max(0.0, confidence))

def match_sms_to_bank(df_entities_with_sms, df_bank, confidence_threshold=0.75):
    """
    Match SMS to bank transactions
    Uses extracted date from SMS timestamp
    """
    matched_pairs = []
    unmatched_sms = []
    
    # Prepare bank data
    df_bank_copy = df_bank.copy()
    df_bank_copy['Transaction_Date'] = pd.to_datetime(df_bank_copy['Transaction_Date'], errors='coerce')
    
    for idx, sms_row in df_entities_with_sms.iterrows():
        if pd.isna(sms_row['amount']):
            unmatched_sms.append({'sms_idx': idx, 'reason': 'no_amount'})
            continue
        
        # Ensure SMS date is datetime
        sms_date = pd.to_datetime(sms_row['date']) if pd.notna(sms_row.get('date')) else None
        if sms_date is None:
            unmatched_sms.append({'sms_idx': idx, 'reason': 'no_date'})
            continue
        
        # Find candidates (same amount, within 2 days, matching date range)
        date_lower = sms_date - timedelta(days=2)
        date_upper = sms_date + timedelta(days=2)
        
        candidates = df_bank_copy[
            ((abs(df_bank_copy['Amount'] - sms_row['amount']) < 5) |
             (abs(df_bank_copy['Amount'] - sms_row['amount']) / sms_row['amount'] < 0.05)) &
            (df_bank_copy['Transaction_Date'] >= date_lower) &
            (df_bank_copy['Transaction_Date'] <= date_upper)
        ]
        
        if len(candidates) == 0:
            unmatched_sms.append({'sms_idx': idx, 'reason': 'no_amount_match'})
            continue
        
        # Score all candidates
        candidates_scored = []
        for bank_idx, bank_row in candidates.iterrows():
            confidence = calculate_match_confidence(sms_row, bank_row)
            candidates_scored.append((bank_idx, confidence))
        
        # Select best match
        best_bank_idx, best_confidence = max(candidates_scored, key=lambda x: x[1])
        
        if best_confidence >= confidence_threshold:
            matched_pairs.append({
                'sms_idx': idx,
                'bank_idx': best_bank_idx,
                'confidence': best_confidence,
                'sms_amount': sms_row['amount'],
                'bank_amount': df_bank_copy.loc[best_bank_idx, 'Amount']
            })
        else:
            unmatched_sms.append({'sms_idx': idx, 'reason': 'low_confidence', 'confidence': best_confidence})
    
    return matched_pairs, unmatched_sms

# Execute Phase 4
print("="*60)
print("PHASE 4: TRANSACTION MATCHING")
print("="*60)

# Filter SMS: only TRANSACTION and FAILED_TRANSACTION
df_to_match = df_entities[
    (df_entities['classification'].isin(['TRANSACTION', 'FAILED_TRANSACTION'])) &
    (df_entities['amount'].notna())
].copy()

# Add SMS data
df_to_match_full = df_to_match.copy()
df_to_match_full['date'] = df_sms_profile.loc[df_to_match.index, 'date'].values
df_to_match_full['body'] = df_sms_profile.loc[df_to_match.index, 'body'].values

matched_pairs, unmatched_sms = match_sms_to_bank(df_to_match_full, df_bank, confidence_threshold=0.75)

print(f"\n✓ Matched {len(matched_pairs)} SMS to bank transactions")
print(f"✗ Unmatched {len(unmatched_sms)} SMS")
print(f"  Matching Rate: {len(matched_pairs)/(len(matched_pairs)+len(unmatched_sms))*100:.1f}%")

print("\nMatched Pairs Sample:")
if matched_pairs:
    for pair in matched_pairs[:5]:
        print(f"  SMS Amount: ₹{pair['sms_amount']:.2f} → Bank Amount: ₹{pair['bank_amount']:.2f} (confidence: {pair['confidence']:.2f})")

# Create matched pairs dataframe
df_matched = pd.DataFrame(matched_pairs)
df_unmatched = pd.DataFrame(unmatched_sms)

print(f"\nConfidence Distribution (Matched):")
if len(df_matched) > 0:
    print(df_matched['confidence'].describe())


PHASE 4: TRANSACTION MATCHING

✓ Matched 2 SMS to bank transactions
✗ Unmatched 9 SMS
  Matching Rate: 18.2%

Matched Pairs Sample:
  SMS Amount: ₹75000.00 → Bank Amount: ₹75000.00 (confidence: 0.84)
  SMS Amount: ₹75000.00 → Bank Amount: ₹75000.00 (confidence: 0.84)

Confidence Distribution (Matched):
count    2.00
mean     0.84
std      0.00
min      0.84
25%      0.84
50%      0.84
75%      0.84
max      0.84
Name: confidence, dtype: float64


In [66]:
# Normalize dates before matching
df_to_match_full['date'] = pd.to_datetime(
    df_to_match_full['date'],
    errors='coerce'
).dt.normalize()

df_bank['Transaction_Date'] = pd.to_datetime(
    df_bank['Transaction_Date'],
    errors='coerce'
).dt.normalize()


# VERIFY: Show date usage in matching
print("\n📅 DATE-BASED MATCHING VERIFICATION")
print("="*60)

if len(matched_pairs) > 0:
    print("\nDate Matching Results (First 10 matches):")
    for i, pair in enumerate(matched_pairs[:10]):
        sms_idx = pair['sms_idx']
        bank_idx = pair['bank_idx']
        
        sms_date = df_to_match_full.loc[sms_idx, 'date']
        bank_date = df_bank.loc[bank_idx, 'Transaction_Date']
        date_diff = abs((sms_date - bank_date).days)
        
        sms_amount = pair['sms_amount']
        bank_amount = pair['bank_amount']
        
        print(f"\n  Match #{i+1}:")
        print(f"    SMS Date: {sms_date.strftime('%Y-%m-%d')} | Bank Date: {bank_date.strftime('%Y-%m-%d')} | Diff: {date_diff} days")
        print(f"    SMS Amount: ₹{sms_amount:.2f} | Bank Amount: ₹{bank_amount:.2f}")
        print(f"    Confidence: {pair['confidence']:.3f}")
else:
    print("⚠️ No matched pairs found. Check date extraction and matching logic.")

# Statistics
print(f"\n\nMatching Date Statistics:")
if len(matched_pairs) > 0:
    date_diffs = []
    for pair in matched_pairs:
        sms_idx = pair['sms_idx']
        bank_idx = pair['bank_idx']
        sms_date = df_to_match_full.loc[sms_idx, 'date']
        bank_date = df_bank.loc[bank_idx, 'Transaction_Date']
        date_diff = abs((sms_date - bank_date).days)
        date_diffs.append(date_diff)
    
    print(f"  Same day matches: {sum(1 for d in date_diffs if d == 0)}")
    print(f"  ±1 day matches: {sum(1 for d in date_diffs if d == 1)}")
    print(f"  ±2 days matches: {sum(1 for d in date_diffs if d == 2)}")
    print(f"  Average date difference: {sum(date_diffs)/len(date_diffs):.2f} days")



📅 DATE-BASED MATCHING VERIFICATION

Date Matching Results (First 10 matches):

  Match #1:
    SMS Date: 2026-02-24 | Bank Date: 2026-02-25 | Diff: 1 days
    SMS Amount: ₹75000.00 | Bank Amount: ₹75000.00
    Confidence: 0.840

  Match #2:
    SMS Date: 2026-03-24 | Bank Date: 2026-03-25 | Diff: 1 days
    SMS Amount: ₹75000.00 | Bank Amount: ₹75000.00
    Confidence: 0.840


Matching Date Statistics:
  Same day matches: 0
  ±1 day matches: 2
  ±2 days matches: 0
  Average date difference: 1.00 days



# PHASE 5: OUTPUT GENERATION
**Objective:** Generate final `true_financial_sms.csv`

**Output Schema:** transaction_date, amount, direction, merchant, account, upi_id, sms_body, confidence


In [67]:

# PHASE 5: GENERATE OUTPUT CSV

def generate_financial_csv(matched_pairs, df_entities, df_sms_profile, df_bank):
    """Generate true_financial_sms.csv"""
    
    output_rows = []
    
    for pair in matched_pairs:
        sms_idx = pair['sms_idx']
        bank_idx = pair['bank_idx']
        
        # Get data
        sms_entity = df_entities.iloc[sms_idx]
        sms_text = df_sms_profile.iloc[sms_idx]
        bank_txn = df_bank.iloc[bank_idx]
        
        # Use bank transaction as primary source of truth, SMS for enrichment
        output_rows.append({
            'transaction_date': bank_txn.get('Transaction_Date', None),
            'amount': bank_txn.get('Amount', None),
            'direction': bank_txn.get('DR/CR_Indicator', sms_entity['direction']),
            'merchant': sms_entity['merchant_normalized'] or bank_txn.get('Recipient_Name', 'UNKNOWN'),
            'account': sms_entity['account'],
            'upi_id': sms_entity['upi_id'],
            'transaction_id': sms_entity['transaction_id'],
            'sms_body': sms_text.get('body', ''),
            'sms_date': sms_text.get('date', None),
            'bank_txn_id': bank_idx,
            'sms_id': sms_idx,
            'confidence_score': pair['confidence'],
            'classification': sms_entity['classification']
        })
    
    df_output = pd.DataFrame(output_rows)
    return df_output

# Execute Phase 5
print("="*60)
print("PHASE 5: OUTPUT GENERATION")
print("="*60)

df_true_financial = generate_financial_csv(matched_pairs, df_entities, df_sms_profile, df_bank)

print(f"\n✓ Generated {len(df_true_financial)} true financial transactions")
print("\nOutput Schema:")
print(df_true_financial.dtypes)

print("\nSample Output (first 5 rows):")
display(df_true_financial[['transaction_date', 'amount', 'direction', 'merchant', 'confidence_score']].head())

# Data Quality Check
print("\n📋 DATA QUALITY CHECK:")
print(f"  ✓ Null values by column:")
print(df_true_financial.isnull().sum())

print(f"\n  ✓ Amount statistics:")
print(df_true_financial['amount'].describe())

print(f"\n  ✓ Confidence statistics:")
print(df_true_financial['confidence_score'].describe())

# Save output
output_path = r'CSVS\true_financial_sms.csv'
df_true_financial.to_csv(output_path, index=False)
print(f"\n✅ Saved to {output_path}")

# # Also save as Excel for readability
# output_excel = r'CSVS\true_financial_sms.xlsx'
# df_true_financial.to_excel(output_excel, index=False)
# print(f"✅ Also saved to {output_excel}")


PHASE 5: OUTPUT GENERATION

✓ Generated 2 true financial transactions

Output Schema:
transaction_date                  datetime64[ns]
amount                                   float64
direction                                 object
merchant                                  object
account                                   object
upi_id                                    object
transaction_id                            object
sms_body                                  object
sms_date            datetime64[ns, Asia/Kolkata]
bank_txn_id                                int64
sms_id                                     int64
confidence_score                         float64
classification                            object
dtype: object

Sample Output (first 5 rows):


,transaction_date,amount,direction,merchant,confidence_score
0,2026-02-25,75000.0,CR,UTR IN,0.84
1,2026-03-25,75000.0,CR,UTR IN,0.84



📋 DATA QUALITY CHECK:
  ✓ Null values by column:
transaction_date    0
amount              0
direction           0
merchant            0
account             2
upi_id              2
transaction_id      2
sms_body            0
sms_date            0
bank_txn_id         0
sms_id              0
confidence_score    0
classification      0
dtype: int64

  ✓ Amount statistics:
count        2.0
mean     75000.0
std          0.0
min      75000.0
25%      75000.0
50%      75000.0
75%      75000.0
max      75000.0
Name: amount, dtype: float64

  ✓ Confidence statistics:
count    2.00
mean     0.84
std      0.00
min      0.84
25%      0.84
50%      0.84
75%      0.84
max      0.84
Name: confidence_score, dtype: float64

✅ Saved to CSVS\true_financial_sms.csv


In [68]:
df_bank.columns

Index(['Transaction_Date', 'Debit', 'Credit', 'Balance', 'Transaction_Mode',
       'DR/CR_Indicator', 'Transaction_ID', 'Recipient_Name', 'Bank', 'UPI_ID',
       'Note', 'Amount'],
      dtype='object')

In [69]:
df_true_financial.columns

Index(['transaction_date', 'amount', 'direction', 'merchant', 'account',
       'upi_id', 'transaction_id', 'sms_body', 'sms_date', 'bank_txn_id',
       'sms_id', 'confidence_score', 'classification'],
      dtype='object')


# PHASE 6: VALIDATION & RECONCILIATION
**Objective:** Validate output against bank statement and check data quality


In [70]:

# PHASE 6: VALIDATION & RECONCILIATION

def validate_output(df_output, df_bank):
    """
    Validate output and compare to bank statement
    """
    
    validation_report = {}
    
    # 1. Schema validation
    required_columns = ['transaction_date', 'amount', 'direction', 'merchant', 'confidence_score']
    validation_report['schema'] = {
        'required_columns_present': all(col in df_output.columns for col in required_columns),
        'total_columns': len(df_output.columns)
    }
    
    # 2. Data quality
    validation_report['data_quality'] = {
        'total_rows': len(df_output),
        'null_count': df_output.isnull().sum().to_dict(),
        'duplicate_rows': df_output.duplicated().sum(),
    }
    
    # 3. Amount validation
    validation_report['amounts'] = {
        'all_positive': (df_output['amount'] > 0).all(),
        'total_amount': df_output['amount'].sum(),
        'min_amount': df_output['amount'].min(),
        'max_amount': df_output['amount'].max(),
    }
    
    # 4. Direction validation
    validation_report['directions'] = {
        'valid_directions': set(df_output['direction'].unique()),
        'credit_count': (df_output['direction'] == 'CR').sum(),
        'debit_count': (df_output['direction'] == 'DR').sum(),
    }
    
    # 5. Reconciliation with bank statement
    matched_amount = df_output[df_output['bank_txn_id'].notna()]['amount'].sum()
    bank_total_amount = df_bank['Amount'].sum()
    reconciliation_rate = (matched_amount / bank_total_amount * 100) if bank_total_amount > 0 else 0
    
    unmatched_bank_txns = len(df_bank) - df_output['bank_txn_id'].nunique()
    
    validation_report['reconciliation'] = {
        'matched_transactions': df_output['bank_txn_id'].nunique(),
        'total_bank_transactions': len(df_bank),
        'unmatched_transactions': unmatched_bank_txns,
        'reconciliation_rate_pct': round(reconciliation_rate, 2),
        'matched_amount': matched_amount,
        'bank_total_amount': bank_total_amount,
    }
    
    # 6. Confidence score distribution
    validation_report['confidence_scores'] = {
        'mean': df_output['confidence_score'].mean(),
        'median': df_output['confidence_score'].median(),
        'min': df_output['confidence_score'].min(),
        'max': df_output['confidence_score'].max(),
        'high_confidence_count': (df_output['confidence_score'] >= 0.90).sum(),
        'medium_confidence_count': ((df_output['confidence_score'] >= 0.75) & (df_output['confidence_score'] < 0.90)).sum(),
        'low_confidence_count': (df_output['confidence_score'] < 0.75).sum(),
    }
    
    return validation_report

def find_unmatched_bank_txns(df_output, df_bank):
    """Find bank transactions without SMS match"""
    matched_bank_indices = set(df_output['bank_txn_id'].dropna().unique())
    unmatched_indices = [i for i in df_bank.index if i not in matched_bank_indices]
    return df_bank.iloc[unmatched_indices]

# Execute Phase 6
print("="*60)
print("PHASE 6: VALIDATION & RECONCILIATION")
print("="*60)

validation_report = validate_output(df_true_financial, df_bank)

print("\n📋 VALIDATION REPORT:\n")
for section, details in validation_report.items():
    print(f"{'='*50}")
    print(f"{section.upper()}")
    print(f"{'='*50}")
    for key, value in details.items():
        if isinstance(value, dict):
            for k, v in value.items():
                print(f"  {k}: {v}")
        else:
            print(f"  {key}: {value}")

print(f"\n{'='*60}")
print("RECONCILIATION SUMMARY")
print(f"{'='*60}")

reconciliation = validation_report['reconciliation']
print(f"✓ Matched: {reconciliation['matched_transactions']} / {reconciliation['total_bank_transactions']} transactions")
print(f"✗ Unmatched: {reconciliation['unmatched_transactions']} transactions")
print(f"📊 Reconciliation Rate: {reconciliation['reconciliation_rate_pct']}%")
print(f"💰 Matched Amount: ₹{reconciliation['matched_amount']:,.2f} / ₹{reconciliation['bank_total_amount']:,.2f}")

# Find unmatched bank transactions
df_unmatched_bank = find_unmatched_bank_txns(df_true_financial, df_bank)

print(f"\n⚠️ UNMATCHED BANK TRANSACTIONS ({len(df_unmatched_bank)}):")
if len(df_unmatched_bank) > 0:
    display(df_unmatched_bank[['Transaction_Date', 'Amount', 'Recipient_Name', 'Description']].head(10))
    
    # Export unmatched for review
    df_unmatched_bank.to_csv(r'CSVS\unmatched_bank_transactions.csv', index=False)
    print(f"✓ Exported to CSVS/unmatched_bank_transactions.csv")

# Save validation report
import json
with open(r'CSVS\validation_report.json', 'w') as f:
    # Convert complex types to strings for JSON serialization
    report_clean = {}
    for section, details in validation_report.items():
        report_clean[section] = {}
        for key, value in details.items():
            if isinstance(value, set):
                report_clean[section][key] = list(value)
            elif isinstance(value, dict):
                report_clean[section][key] = {str(k): str(v) for k, v in value.items()}
            else:
                report_clean[section][key] = value
    
    json.dump(report_clean, f, indent=2, default=str)

print(f"✓ Validation report saved to CSVS/validation_report.json")


PHASE 6: VALIDATION & RECONCILIATION

📋 VALIDATION REPORT:

SCHEMA
  required_columns_present: True
  total_columns: 13
DATA_QUALITY
  total_rows: 2
  transaction_date: 0
  amount: 0
  direction: 0
  merchant: 0
  account: 2
  upi_id: 2
  transaction_id: 2
  sms_body: 0
  sms_date: 0
  bank_txn_id: 0
  sms_id: 0
  confidence_score: 0
  classification: 0
  duplicate_rows: 0
AMOUNTS
  all_positive: True
  total_amount: 150000.0
  min_amount: 75000.0
  max_amount: 75000.0
DIRECTIONS
  valid_directions: {'CR'}
  credit_count: 2
  debit_count: 0
RECONCILIATION
  matched_transactions: 2
  total_bank_transactions: 500
  unmatched_transactions: 498
  reconciliation_rate_pct: 8735.05
  matched_amount: 150000.0
  bank_total_amount: 1717.2199999999903
CONFIDENCE_SCORES
  mean: 0.8399999999999999
  median: 0.8399999999999999
  min: 0.8399999999999999
  max: 0.8399999999999999
  high_confidence_count: 0
  medium_confidence_count: 2
  low_confidence_count: 0

RECONCILIATION SUMMARY
✓ Matched: 2 / 50

KeyError: "['Description'] not in index"


# PHASE 7: PERFORMANCE METRICS & SUMMARY
**Objective:** Calculate pipeline performance and quality metrics


In [ ]:

# PHASE 7: PERFORMANCE METRICS

import time

def calculate_metrics(df_entities, df_true_financial, df_bank):
    """Calculate all pipeline quality metrics"""
    
    metrics = {}
    
    # 1. Classification Metrics
    total_sms = len(df_entities)
    transaction_sms = (df_entities['classification'] == 'TRANSACTION').sum()
    failed_sms = (df_entities['classification'] == 'FAILED_TRANSACTION').sum()
    otp_sms = (df_entities['classification'] == 'OTP').sum()
    promotional_sms = (df_entities['classification'] == 'PROMOTIONAL').sum()
    spam_sms = (df_entities['classification'] == 'SPAM').sum()
    service_sms = (df_entities['classification'] == 'SERVICE_NOTIFICATION').sum()
    unknown_sms = (df_entities['classification'] == 'UNKNOWN').sum()
    
    metrics['classification'] = {
        'total_sms': total_sms,
        'transaction_sms': transaction_sms,
        'failed_transaction_sms': failed_sms,
        'otp_sms': otp_sms,
        'promotional_sms': promotional_sms,
        'spam_sms': spam_sms,
        'service_sms': service_sms,
        'unknown_sms': unknown_sms,
        'transaction_percentage': round((transaction_sms + failed_sms) / total_sms * 100, 2),
    }
    
    # 2. Entity Extraction Metrics
    metrics['entity_extraction'] = {
        'amount_extraction_rate': round(df_entities['amount'].notna().sum() / total_sms * 100, 2),
        'direction_extraction_rate': round(df_entities['direction'].notna().sum() / total_sms * 100, 2),
        'merchant_extraction_rate': round(df_entities['merchant'].notna().sum() / total_sms * 100, 2),
        'upi_extraction_rate': round(df_entities['upi_id'].notna().sum() / total_sms * 100, 2),
        'account_extraction_rate': round(df_entities['account'].notna().sum() / total_sms * 100, 2),
    }
    
    # 3. Matching Metrics
    metrics['matching'] = {
        'total_matched': len(df_true_financial),
        'total_bank_txns': len(df_bank),
        'matching_rate': round(len(df_true_financial) / len(df_bank) * 100, 2),
        'avg_confidence': round(df_true_financial['confidence_score'].mean(), 4),
        'median_confidence': round(df_true_financial['confidence_score'].median(), 4),
    }
    
    # 4. Confidence Distribution
    high_conf = (df_true_financial['confidence_score'] >= 0.90).sum()
    med_conf = ((df_true_financial['confidence_score'] >= 0.75) & (df_true_financial['confidence_score'] < 0.90)).sum()
    low_conf = (df_true_financial['confidence_score'] < 0.75).sum()
    
    metrics['confidence_distribution'] = {
        'high_confidence_90plus': high_conf,
        'medium_confidence_75to90': med_conf,
        'low_confidence_below75': low_conf,
    }
    
    # 5. Financial Accuracy
    matched_amount = df_true_financial['amount'].sum()
    bank_amount = df_bank['Amount'].sum()
    
    metrics['financial_accuracy'] = {
        'total_matched_amount': round(matched_amount, 2),
        'total_bank_amount': round(bank_amount, 2),
        'reconciliation_coverage': round(matched_amount / bank_amount * 100, 2),
    }
    
    return metrics

print("="*60)
print("PHASE 7: PERFORMANCE METRICS")
print("="*60)

metrics = calculate_metrics(df_entities, df_true_financial, df_bank)

print("\n📊 CLASSIFICATION METRICS:")
for k, v in metrics['classification'].items():
    print(f"  {k}: {v}")

print("\n🎯 ENTITY EXTRACTION RATES:")
for k, v in metrics['entity_extraction'].items():
    print(f"  {k}: {v}%")

print("\n🔗 MATCHING METRICS:")
for k, v in metrics['matching'].items():
    print(f"  {k}: {v}")

print("\n📈 CONFIDENCE DISTRIBUTION:")
for k, v in metrics['confidence_distribution'].items():
    print(f"  {k}: {v}")

print("\n💰 FINANCIAL ACCURACY:")
for k, v in metrics['financial_accuracy'].items():
    print(f"  {k}: ₹{v:,.2f}" if 'amount' in k else f"  {k}: {v}%")

# Export metrics
with open(r'CSVS\pipeline_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(f"\n✅ Metrics exported to CSVS/pipeline_metrics.json")

# Create summary visualization
print("\n" + "="*60)
print("PIPELINE EXECUTION SUMMARY")
print("="*60)
print(f"""
INPUT:
  ├─ SMS Count: {len(df_sms_profile):,}
  └─ Bank Transactions: {len(df_bank)}

OUTPUT:
  ├─ Classified SMS: {len(df_entities)}
  ├─ Matched Transactions: {len(df_true_financial)}
  ├─ Reconciliation Rate: {metrics['financial_accuracy']['reconciliation_coverage']}%
  └─ Avg Confidence: {metrics['matching']['avg_confidence']:.2f}

QUALITY GATES:
  ✓ Entity Extraction: {metrics['entity_extraction']['amount_extraction_rate']}% (Target: ≥95%)
  ✓ Matching Rate: {metrics['matching']['matching_rate']}% (Target: ≥85%)
  ✓ High Confidence: {metrics['confidence_distribution']['high_confidence_90plus']} (Target: ≥80%)

FILES GENERATED:
  ✓ true_financial_sms.csv
  ✓ true_financial_sms.xlsx
  ✓ unmatched_bank_transactions.csv
  ✓ validation_report.json
  ✓ pipeline_metrics.json
""")
